# Ollama Local API Example
This notebook demonstrates how to call a local Ollama server using configuration from a `.env` file.

In [2]:
# Install dependencies if running in a fresh environment
# %pip install -r ../requirements.txt

In [3]:
import os
from dotenv import load_dotenv
import requests

# Load environment variables from .env file
load_dotenv(override=True)
OLLAMA_URL = os.getenv('OLLAMA_URL')
OLLAMA_API_KEY = os.getenv('OLLAMA_API_KEY')
print(f'Using Ollama URL: {OLLAMA_URL}')

Using Ollama URL: http://localhost:11434


In [12]:
# Example: Make a POST request to Ollama local API with system and user prompts, rendering markdown output
import json
from IPython.display import Markdown, display
headers = {
    'Authorization': f'Bearer {OLLAMA_API_KEY}' if OLLAMA_API_KEY else ''
}
data = {
    'model': 'mistral',
    'messages': [
        {
            'role': 'system',
            'content': 'You are an assistant that analyses the contents of a website, and provides a short summary , ignoring text that might navigation related. Respond in markdown text.'
        },
        {
            'role': 'user',
            'content': 'Summarize the main content of https://shiva-adigopula.vercel.app/'
        }
    ]
}
try:
    full_message = ''
    with requests.post(f'{OLLAMA_URL}/api/generate', json=data, headers=headers, stream=True) as response:
        response.raise_for_status()
        for line in response.iter_lines():
            if line:
                chunk = json.loads(line.decode())
                if 'response' in chunk:
                    full_message += chunk['response']
                elif 'message' in chunk:
                    full_message += chunk['message']
    display(Markdown(full_message))
except Exception as e:
    print('Error calling Ollama:', e)

In [ ]:
# Fetch website content and summarize using Ollama
import requests
from bs4 import BeautifulSoup
import json
from IPython.display import Markdown, display

# Fetch the website
url = "https://shiva-adigopula.vercel.app/"
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')

# Remove navigation and footer elements (common tags)
for tag in soup.find_all(['nav', 'footer', 'header', 'aside']):
    tag.decompose()

# Get visible text
page_text = ' '.join(soup.stripped_strings)

# Truncate if too long for the model
max_length = 2000  # adjust as needed
doc_text = page_text[:max_length]

headers = {
    'Authorization': f'Bearer {OLLAMA_API_KEY}' if OLLAMA_API_KEY else ''
}
data = {
    'model': 'mistral',
    'messages': [
        {
            'role': 'system',
            'content': 'You are an assistant that analyses the contents of a website, and provides a short summary, ignoring text that might be navigation related. Respond in markdown text.'
        },
        {
            'role': 'user',
            'content': f'Here is the website content:\n"""{doc_text}"""\nPlease summarize.'
        }
    ]
}
try:
    full_message = ''
    with requests.post(f'{OLLAMA_URL}/api/generate', json=data, headers=headers, stream=True) as resp:
        resp.raise_for_status()
        for line in resp.iter_lines():
            if line:
                chunk = json.loads(line.decode())
                if 'response' in chunk:
                    full_message += chunk['response']
                elif 'message' in chunk:
                    full_message += chunk['message']
    display(Markdown(full_message))
except Exception as e:
    print('Error calling Ollama:', e)
